# Space X Falcon 9 First Stage Landing Prediction
## Lab 6: Launch sites locations analysis with Folium

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [2]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
                        "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [3]:
launch_sites_df = spacex_df[['Launch Site', 'Lat', 'Long']].groupby(
    ['Launch Site'], as_index=False).first()
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### TASK 1: Mark all launch sites on a map

In [4]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(
    folium.Popup('NASA Johnson Space Center')).add_to(site_map)
folium.map.Marker(nasa_coordinate, icon=DivIcon(
    icon_size=(20, 20), icon_anchor=(0, 0),
    html='<div style="font-size: 12; color:#d35400;"><b>NASA JSC</b></div>')).add_to(site_map)

for _, r in launch_sites_df.iterrows():
    coordinate = [r['Lat'], r['Long']]
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(
        folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker(coordinate, icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{r["Launch Site"]}</b></div>'
    )).add_to(site_map)

site_map

**Observation:** every launch site is close to the coast and relatively close to
the equator. Proximity to the equator gives the rocket extra speed from the Earth's
rotation, and proximity to the sea means a failed launch falls over water.

### TASK 2: Mark the success/failed launches for each site on the map

In [5]:
spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda c: 'green' if c == 1 else 'red')
spacex_df.tail(10)

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long,marker_color
46,43,2017-10-11,22:53:00,F9 FT B1031.2,KSC LC-39A,SES-11 / EchoStar 105,5200.00,GTO,SES EchoStar,Success (drone ship),1,28.573255,-80.646895,green
47,44,2017-10-30,19:34:00,F9 B4 B1042.1,KSC LC-39A,Koreasat 5A,3500.00,GTO,KT Corporation,Success (drone ship),1,28.573255,-80.646895,green
48,54,2018-05-11,20:14:00,F9 B5 B1046.1,KSC LC-39A,Bangabandhu-1,3600.00,GTO,Thales-Alenia/BTRC,Success (drone ship),1,28.573255,-80.646895,green
49,45,2017-12-15,15:36:00,F9 FT B1035.2,CCAFS SLC-40,SpaceX CRS-13,2205.00,LEO (ISS),NASA (CRS),Success (ground pad),1,28.563197,-80.576820,green
50,47,2018-01-08,1:00:00,F9 B4 B1043.1,CCAFS SLC-40,Zuma,3696.65,LEO,Northrop Grumman,Success (ground pad),1,28.563197,-80.576820,green
51,48,2018-01-31,21:25:00,F9 FT B1032.2,CCAFS SLC-40,GovSat-1 / SES-16,4230.00,GTO,SES,Controlled (ocean),0,28.563197,-80.576820,red
52,50,2018-03-06,5:33:00,F9 B4 B1044,CCAFS SLC-40,Hispasat 30W-6 PODSat,6092.00,GTO,Hispasat NovaWurks,No attempt,0,28.563197,-80.576820,red
53,52,2018-04-02,20:30:00,F9 B4 B1039.2,CCAFS SLC-40,SpaceX CRS-14,2647.00,LEO (ISS),NASA (CRS),No attempt,0,28.563197,-80.576820,red
54,53,2018-04-18,22:51:00,F9 B4 B1045.1,CCAFS SLC-40,Transiting Exoplanet Survey Satellite (TESS),362.00,HEO,NASA (LSP),Success (drone ship),1,28.563197,-80.576820,green
55,56,2018-06-04,4:45:00,F9 B4 B1040.2,CCAFS SLC-40,SES-12,5384.00,GTO,SES,No attempt,0,28.563197,-80.576820,red


In [6]:
site_map = folium.Map(location=[31.0, -95.0], zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map)

for _, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f'{record["Launch Site"]} — {"Success" if record["class"] == 1 else "Failure"}'
    ).add_to(marker_cluster)

MousePosition(position='topright', separator=' Long: ', prefix='Lat:').add_to(site_map)
site_map

In [7]:
success_rate = spacex_df.groupby('Launch Site')['class'].agg(['sum', 'count', 'mean'])
success_rate.columns = ['Successes', 'Launches', 'Success rate']
success_rate.sort_values('Success rate', ascending=False)

,Successes,Launches,Success rate
Launch Site,,,
KSC LC-39A,10,13,0.769231
CCAFS SLC-40,3,7,0.428571
VAFB SLC-4E,4,10,0.400000
CCAFS LC-40,7,26,0.269231


### TASK 3: Calculate the distances between a launch site to its proximities

In [8]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [9]:
launch_site_lat, launch_site_lon = 28.56319, -80.57683      # CCAFS SLC-40
proximities = {'Coastline': (28.56367, -80.57163),
               'Railway': (28.57209, -80.58527),
               'Highway': (28.56335, -80.57085),
               'Titusville city': (28.61208, -80.80764)}

for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    print(f"{name:16s} {d:8.2f} km")

Coastline            0.51 km
Railway              1.29 km
Highway              0.58 km
Titusville city     23.19 km


In [10]:
site_map = folium.Map(location=[28.5645, -80.5750], zoom_start=15)
launch_site = [launch_site_lat, launch_site_lon]

folium.Circle(launch_site, radius=200, color='#000000', fill=True).add_child(
    folium.Popup('CCAFS SLC-40')).add_to(site_map)
folium.map.Marker(launch_site, icon=DivIcon(
    icon_size=(20, 20), icon_anchor=(0, 0),
    html='<div style="font-size: 12; color:#000000;"><b>CCAFS SLC-40</b></div>'
)).add_to(site_map)

for (name, (lat, lon)), colour in zip(list(proximities.items())[:3],
                                      ['#2a9d8f', '#264653', '#e76f51']):
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    folium.Marker([lat, lon], icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size: 12; color:{colour};"><b>{name}: {d:.2f} KM</b></div>'
    )).add_to(site_map)
    folium.PolyLine([launch_site, [lat, lon]], weight=2, color=colour).add_to(site_map)

site_map

**Findings**

* Launch sites are very close to the coastline (0.51 km) — failed launches fall into the sea.
* They are close to a highway (0.58 km) and a railway (1.29 km), which is needed to move
  rocket stages and propellant to the pad.
* They keep a safe distance from cities: the nearest, Titusville, is 23.19 km away.